In [58]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_log_error
import joblib
from pathlib import Path


In [59]:
import sys, os
sys.path.append(os.path.abspath(".."))

print("Added to sys.path:", os.path.abspath(".."))


Added to sys.path: C:\Users\PC Center\Desktop\DSP_Assignment


In [60]:
from house_prices.train import build_model
import pandas as pd

training_df = pd.read_csv("../data/train.csv")
model_performance_dict = build_model(training_df)
print(model_performance_dict)


{'rmsle': np.float64(0.24881)}


In [61]:
def compute_rmsle(y_true, y_pred, precision=5):
    """Compute Root Mean Squared Logarithmic Error (RMSLE)."""
    rmsle = np.sqrt(mean_squared_log_error(y_true, np.maximum(y_pred, 0)))
    return round(rmsle, precision)


In [62]:
import os
print(os.getcwd())


C:\Users\PC Center\Desktop\DSP_Assignment\notebooks


In [63]:
# Load data
df = pd.read_csv("../data/train.csv")

# Select features
features_cont = ["GrLivArea", "GarageArea"]
features_cat = ["MSZoning", "HouseStyle"]
target = "SalePrice"

# Split early to avoid data leakage
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Separate features and target
X_train = train_df[features_cont + features_cat]
y_train = train_df[target]
X_test = test_df[features_cont + features_cat]
y_test = test_df[target]
df.head()


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [64]:
import os
print(os.getcwd())

C:\Users\PC Center\Desktop\DSP_Assignment\notebooks


In [65]:
# Scale numeric features
scaler = StandardScaler()
X_train_cont = scaler.fit_transform(X_train[features_cont])
X_test_cont = scaler.transform(X_test[features_cont])

# Encode categorical features
encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
X_train_cat = encoder.fit_transform(X_train[features_cat])
X_test_cat = encoder.transform(X_test[features_cat])

# Combine numeric + categorical
X_train_proc = np.hstack([X_train_cont, X_train_cat])
X_test_proc = np.hstack([X_test_cont, X_test_cat])


In [66]:
# Train model
model = Ridge(alpha=10.0, random_state=42)
model.fit(X_train_proc, y_train)

print("Model training complete.")


Model training complete.


In [67]:
# Predict and evaluate
y_pred = model.predict(X_test_proc)
rmsle_score = compute_rmsle(y_test, y_pred)

print(f"Validation RMSLE: {rmsle_score}")
print("Train R²:", round(model.score(X_train_proc, y_train), 4))
print("Test R²:", round(model.score(X_test_proc, y_test), 4))


Validation RMSLE: 0.24881
Train R²: 0.6407
Test R²: 0.6994


In [68]:
# Create models folder
Path("../models").mkdir(parents=True, exist_ok=True)

# Save model and preprocessing objects
joblib.dump(model, "../models/model.joblib")
joblib.dump(scaler, "../models/scaler.joblib")
joblib.dump(encoder, "../models/encoder.joblib")

print("Saved model, scaler, and encoder.")


Saved model, scaler, and encoder.


In [69]:

model_loaded = joblib.load("../models/model.joblib")
scaler_loaded = joblib.load("../models/scaler.joblib")
encoder_loaded = joblib.load("../models/encoder.joblib")

inference_df = pd.read_csv("../data/train.csv")
for col in features_cont:
    inference_df[col] = inference_df[col].fillna(inference_df[col].median())
for col in features_cat:
    inference_df[col] = inference_df[col].fillna(inference_df[col].mode()[0])


X_inf_cont = scaler_loaded.transform(inference_df[features_cont])
X_inf_cat = encoder_loaded.transform(inference_df[features_cat])
X_inf_proc = np.hstack([X_inf_cont, X_inf_cat])

predictions = model_loaded.predict(X_inf_proc)
print("Predictions:", predictions[:10])


Predictions: [197407.66156834 170420.3877255  210633.26263961 208050.65326573
 271968.7808358  151234.7181657  228014.91180793 224759.80571308
 166876.21128792 115092.13529748]


In [70]:
output = pd.DataFrame({
    "Id": inference_df["Id"],
    "SalePrice": np.maximum(predictions, 0)
})
output.to_csv("../data/house-prices/inference_predictions.csv", index=False)
print("Predictions saved to CSV.")


Predictions saved to CSV.
